In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.regularizers import l2
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

In [2]:
#Import Data
df = pd.read_csv('final_dataset.csv')
df 

,Unnamed: 0,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTGS,ATGS,HTGC,...,HTLossStreak3,HTLossStreak5,ATWinStreak3,ATWinStreak5,ATLossStreak3,ATLossStreak5,HTGD,ATGD,DiffPts,DiffFormPts
0,0,19/08/00,Charlton,Man City,4,0,H,0,0,0,...,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000
1,1,19/08/00,Chelsea,West Ham,4,2,H,0,0,0,...,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000
2,2,19/08/00,Coventry,Middlesbrough,1,3,NH,0,0,0,...,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000
3,3,19/08/00,Derby,Southampton,2,2,NH,0,0,0,...,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000
4,4,19/08/00,Leeds,Everton,2,0,H,0,0,0,...,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6835,6835,13/05/18,Newcastle,Chelsea,3,0,H,36,62,47,...,0,0,1,0,0,0,-0.289474,0.710526,-0.763158,-0.263158
6836,6836,13/05/18,Southampton,Man City,0,1,NH,37,105,55,...,0,0,1,0,0,0,-0.473684,2.052632,-1.605263,-0.131579
6837,6837,13/05/18,Swansea,Stoke,1,2,NH,27,33,54,...,0,0,0,0,0,0,-0.710526,-0.894737,0.078947,-0.052632
6838,6838,13/05/18,Tottenham,Leicester,5,4,H,69,52,32,...,0,0,0,0,0,0,0.973684,-0.078947,0.710526,0.078947


In [3]:
#Feature Engineering
def determine_outcome(row):
    if row['FTHG'] > row['FTAG']:
        return 1
    elif row['FTHG'] < row['FTAG']:
        return 2
    else:
        return 0
    
df['outcome'] = df.apply(determine_outcome, axis=1)

In [4]:
#Set target variable
features = ['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'HTGS', 'ATGS', 'HTGC', 'HTFormPtsStr',
            'ATFormPtsStr', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5',
            'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5']
target = ['outcome']

data_encoded = pd.get_dummies(df[features])
data_model = pd.concat([data_encoded, df[target]], axis=1)

X = data_model.drop(columns=target)
y = data_model[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Encode and one-hot encode the labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train.values.ravel())
y_test_encoded = label_encoder.transform(y_test.values.ravel())
y_train_categorical = to_categorical(y_train_encoded)
y_test_categorical = to_categorical(y_test_encoded)

In [5]:
# DNN Model
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.01)))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Dense(3, activation='softmax'))  # Assuming 3 classes for 'outcome'

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train_scaled, y_train_categorical, epochs=100, validation_split=0.2, batch_size=32, callbacks=[early_stopping])

# Plot training & validation loss and accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

Epoch 1/100


/Users/jackcook/miniconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [1]:
#Calibration Plot 
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

# Predict probabilities on the test set using the original model
y_pred_prob_test = model.predict(X_test_scaled)

# Initialize a figure for the calibration plots
plt.figure(figsize=(15, 10))

# Loop through each class and plot the calibration curve
for i in range(y_pred_prob_test.shape[1]):
    # Extract the predicted probabilities for class i
    y_pred_prob_class = y_pred_prob_test[:, i]
    
    # Train a logistic regression model on the predicted probabilities
    calibration_model = LogisticRegression()
    calibration_model.fit(y_pred_prob_class.reshape(-1, 1), (y_test_encoded == i).astype(int))
    
    # Calibrate the predicted probabilities using the logistic regression model
    y_pred_prob_calibrated = calibration_model.predict_proba(y_pred_prob_class.reshape(-1, 1))[:, 1]
    
    # Compute and plot the calibration curve for the calibrated model
    prob_true_calibrated, prob_pred_calibrated = calibration_curve((y_test_encoded == i).astype(int), y_pred_prob_calibrated, n_bins=10, strategy='uniform')
    
    plt.plot(prob_pred_calibrated, prob_true_calibrated, marker='o', label=f'Class {i}')
    
# Add a perfectly calibrated line
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated')

# Finalize the plot
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Plots for Each Class')
plt.legend()
plt.show()



NameError: name 'model' is not defined

In [ ]:
#Brier Score Validation Tests

from sklearn.metrics import brier_score_loss

# Predict probabilities on the test set using the original model
y_pred_prob_test = model.predict(X_test_scaled)

# Initialize a list to store the Brier scores for each class
brier_scores = []

# Loop through each class and compute the Brier score
for i in range(y_pred_prob_test.shape[1]):
    # Extract the true binary labels for class i
    y_true_class = (y_test_encoded == i).astype(int)
    
    # Extract the predicted probabilities for class i
    y_pred_prob_class = y_pred_prob_test[:, i]
    
    # Compute the Brier score for class i
    brier_score = brier_score_loss(y_true_class, y_pred_prob_class)
    
    # Append the Brier score to the list
    brier_scores.append(brier_score)
    
    print(f'Brier score for class {i}: {brier_score}')

# Compute the average Brier score across all classes
average_brier_score = np.mean(brier_scores)
print(f'Average Brier score: {average_brier_score}')